# 03. SW Regression Story
## sw-vendor-v1.2.3 → v1.3.0 업그레이드의 KPI 영향

> **Demo 목표**: 30초 안에 *"LLC 버그가 있었고, 수정했고, 전력이 줄었다"* 를 데이터로 증명

| | sw-vendor-v1.2.3 | sw-vendor-v1.3.0 |
|--|--|--|
| LLC 모드 | `shared` (버그) | `per_ip_partition` (수정) |
| Feasibility | `exploration_only` | `production_ready` |
| Total Power | 2,350 mW | 2,150 mW (**-8.5%**) |
| Gate Result | WARN | **PASS** |

---

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import numpy as np
import plotly.graph_objects as go
from IPython.display import display, HTML

from demo.notebooks.utils.db_connection import get_engine, query_df
from demo.notebooks.utils.plot_theme import (
    apply_theme, FEASIBILITY_COLORS, GATE_COLORS,
    SW_VERSION_PALETTE, ACCENT, BG_LIGHT, plotly_layout,
)

apply_theme()
ENGINE = get_engine()

# 이 노트북 전체에서 쓸 색상 상수
C_OLD = FEASIBILITY_COLORS['exploration_only']  # #F39C12 주황
C_NEW = FEASIBILITY_COLORS['production_ready']  # #2ECC71 초록
LABELS = {'sw-vendor-v1.2.3': 'v1.2.3 (before)', 'sw-vendor-v1.3.0': 'v1.3.0 (after)'}

print('✓ ready')

---
## Section 1. Feasibility 분포

**질문**: *SW 버전별로 evidence가 어떤 feasibility로 분류되는가?*

In [ ]:
df_feas = query_df("""
    SELECT
        sw_version_hint,
        overall_feasibility,
        COUNT(*)                                     AS cnt,
        AVG((kpi->>'total_power_mw')::float)         AS avg_total_power_mw,
        AVG((kpi->>'peak_power_mw')::float)          AS avg_peak_power_mw,
        AVG((kpi->>'avg_ddr_bw_gbps')::float)        AS avg_ddr_bw_gbps,
        AVG((kpi->>'frame_latency_ms')::float)       AS avg_frame_latency_ms
    FROM evidence
    WHERE kind = 'evidence.simulation'
    GROUP BY sw_version_hint, overall_feasibility
    ORDER BY sw_version_hint
""", ENGINE)

display(df_feas)

In [ ]:
fig1, axes1 = plt.subplots(1, 2, figsize=(12, 3.5),
                            gridspec_kw={'width_ratios': [1, 2]})

# --- 좌: Feasibility stacked bar ---
ax_feas = axes1[0]
pivot_feas = df_feas.pivot(
    index='sw_version_hint', columns='overall_feasibility', values='cnt'
).fillna(0)

sw_labels = [LABELS.get(s, s) for s in pivot_feas.index]
x = np.arange(len(sw_labels))
bottom = np.zeros(len(x))

for col in pivot_feas.columns:
    vals = pivot_feas[col].values
    bars = ax_feas.bar(x, vals, bottom=bottom,
                       color=FEASIBILITY_COLORS.get(col, '#95A5A6'),
                       label=col, width=0.5, edgecolor='white')
    for rect, v, b in zip(bars, vals, bottom):
        if v > 0:
            ax_feas.text(rect.get_x() + rect.get_width()/2,
                         b + v/2, col, ha='center', va='center',
                         fontsize=8, color='white', fontweight='bold')
    bottom += vals

ax_feas.set_xticks(x)
ax_feas.set_xticklabels(sw_labels, fontsize=9)
ax_feas.set_ylabel('Evidence 수')
ax_feas.set_title('Feasibility 분포', fontsize=11, fontweight='bold', color=ACCENT)
ax_feas.set_ylim(0, bottom.max() * 1.4)

# --- 우: 주요 KPI 변화 mini summary ---
ax_kpi = axes1[1]
kpi_cols = ['avg_total_power_mw', 'avg_peak_power_mw',
            'avg_ddr_bw_gbps', 'avg_frame_latency_ms']
kpi_labels = ['Total Power\n(mW)', 'Peak Power\n(mW)',
              'Avg DDR BW\n(GBps)', 'Frame Latency\n(ms)']

v_old = df_feas[df_feas['sw_version_hint']=='sw-vendor-v1.2.3'][kpi_cols].values[0]
v_new = df_feas[df_feas['sw_version_hint']=='sw-vendor-v1.3.0'][kpi_cols].values[0]
pct   = (v_new - v_old) / v_old * 100

x2 = np.arange(len(kpi_labels))
w  = 0.35
ax_kpi.bar(x2 - w/2, v_old, width=w, color=C_OLD, label='v1.2.3 (before)', edgecolor='white')
ax_kpi.bar(x2 + w/2, v_new, width=w, color=C_NEW, label='v1.3.0 (after)',  edgecolor='white')

for i, (vo, vn, p) in enumerate(zip(v_old, v_new, pct)):
    ax_kpi.text(i, max(vo, vn) + max(vo,vn)*0.03,
                f'{p:+.1f}%', ha='center', va='bottom',
                fontsize=9, fontweight='bold',
                color=C_NEW if p < 0 else '#E74C3C')

ax_kpi.set_xticks(x2)
ax_kpi.set_xticklabels(kpi_labels, fontsize=9)
ax_kpi.set_title('KPI 변화 (전체 평균)', fontsize=11, fontweight='bold', color=ACCENT)
ax_kpi.legend(fontsize=9)

fig1.suptitle('SW 업그레이드 효과: v1.2.3 → v1.3.0',
              fontsize=13, fontweight='bold', color=ACCENT, y=1.02)
plt.tight_layout()
plt.savefig('s1_feasibility_kpi.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved: s1_feasibility_kpi.png')

---
## Section 2. KPI Paired Comparison — Dumbbell Chart

**질문**: *동일 variant(UHD60-HDR10-H265)에서 SW 버전만 바꿨을 때 각 KPI가 얼마나 개선됐는가?*

동일 시나리오·variant로 짝지어 비교 → confounding 없이 SW 효과만 분리.

In [ ]:
df_kpi = query_df("""
    SELECT
        variant_ref,
        sw_version_hint,
        overall_feasibility,
        (kpi->>'total_power_mw')::float    AS total_power_mw,
        (kpi->>'peak_power_mw')::float     AS peak_power_mw,
        (kpi->>'avg_ddr_bw_gbps')::float   AS avg_ddr_bw_gbps,
        (kpi->>'peak_ddr_bw_gbps')::float  AS peak_ddr_bw_gbps,
        (kpi->>'frame_latency_ms')::float  AS frame_latency_ms
    FROM evidence
    WHERE kind        = 'evidence.simulation'
      AND scenario_ref = 'uc-camera-recording'
      AND variant_ref  = 'UHD60-HDR10-H265'
    ORDER BY sw_version_hint
""", ENGINE)

# Pivot: KPI × (v1.2.3, v1.3.0)
kpi_metrics = ['total_power_mw','peak_power_mw',
               'avg_ddr_bw_gbps','peak_ddr_bw_gbps','frame_latency_ms']
row_old = df_kpi[df_kpi['sw_version_hint']=='sw-vendor-v1.2.3'].iloc[0]
row_new = df_kpi[df_kpi['sw_version_hint']=='sw-vendor-v1.3.0'].iloc[0]

df_paired = pd.DataFrame({
    'kpi':    kpi_metrics,
    'before': [row_old[m] for m in kpi_metrics],
    'after':  [row_new[m] for m in kpi_metrics],
})
df_paired['delta']   = df_paired['after']  - df_paired['before']
df_paired['delta_pct'] = df_paired['delta'] / df_paired['before'] * 100
df_paired['unit'] = ['mW','mW','GBps','GBps','ms']

display(HTML('<h4>KPI Paired Comparison — UHD60-HDR10-H265</h4>'))
display(
    df_paired.style
    .format({'before':'{:.1f}','after':'{:.1f}',
             'delta':'{:+.1f}','delta_pct':'{:+.1f}%'})
    .map(lambda v: f'color:{C_NEW};font-weight:bold' if isinstance(v,(int,float)) and v < 0
         else (f'color:#E74C3C;font-weight:bold' if isinstance(v,(int,float)) and v > 0 else ''),
         subset=['delta','delta_pct'])
)

In [ ]:
# Dumbbell chart — 정규화 없이 각 KPI별 독립 패널로
n = len(kpi_metrics)
kpi_display = ['Total Power\n(mW)', 'Peak Power\n(mW)',
               'Avg DDR BW\n(GBps)', 'Peak DDR BW\n(GBps)', 'Frame Latency\n(ms)']

fig2, axes2 = plt.subplots(1, n, figsize=(14, 3.8))

for ax, metric, label in zip(axes2, kpi_metrics, kpi_display):
    vb = row_old[metric]
    va = row_new[metric]
    pct = (va - vb) / vb * 100
    improved = va < vb  # 낮을수록 좋은 지표
    arrow_color = C_NEW if improved else '#E74C3C'

    # dumbbell 선
    ax.plot([0, 1], [vb, va], color='#BDC3C7', linewidth=2, zorder=1)

    # before dot
    ax.scatter(0, vb, s=200, color=C_OLD, zorder=3, edgecolors='white', linewidths=2)
    ax.annotate(f'{vb:.1f}', xy=(0, vb), xytext=(-0.18, vb),
                ha='right', va='center', fontsize=10, color=C_OLD, fontweight='bold')

    # after dot
    ax.scatter(1, va, s=200, color=C_NEW, zorder=3, edgecolors='white', linewidths=2)
    ax.annotate(f'{va:.1f}', xy=(1, va), xytext=(1.18, va),
                ha='left', va='center', fontsize=10, color=C_NEW, fontweight='bold')

    # delta 배지
    mid_y = (vb + va) / 2
    ax.text(0.5, mid_y, f'{pct:+.1f}%',
            ha='center', va='center', fontsize=10, fontweight='bold',
            color='white',
            bbox=dict(boxstyle='round,pad=0.3', facecolor=arrow_color, alpha=0.9))

    margin = abs(vb - va) * 0.6 + abs(vb) * 0.05
    ax.set_ylim(min(vb,va) - margin, max(vb,va) + margin)
    ax.set_xlim(-0.4, 1.4)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['v1.2.3', 'v1.3.0'], fontsize=9)
    ax.set_title(label, fontsize=9, fontweight='bold', color=ACCENT)
    ax.tick_params(axis='y', labelsize=8)

# 공통 범례
before_dot = mlines.Line2D([], [], color=C_OLD, marker='o', linestyle='None',
                            markersize=9, label='v1.2.3 before (exploration_only)')
after_dot  = mlines.Line2D([], [], color=C_NEW, marker='o', linestyle='None',
                            markersize=9, label='v1.3.0 after (production_ready)')
fig2.legend(handles=[before_dot, after_dot],
            loc='lower center', ncol=2, fontsize=9,
            bbox_to_anchor=(0.5, -0.08))

fig2.suptitle('KPI Dumbbell Chart — UHD60-HDR10-H265 (동일 variant, SW만 교체)',
              fontsize=12, fontweight='bold', color=ACCENT)
plt.tight_layout()
plt.savefig('s2_dumbbell.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved: s2_dumbbell.png')

---
## Section 3. Root Cause — 어디를 고쳤는가?

**질문**: *이슈의 fix_sw_ref를 따라가면 SW profile에서 정확히 어떤 플래그가 바뀌었는가?*

In [ ]:
# 이슈 상세
df_issue = query_df("""
    SELECT
        id,
        metadata->>'title'              AS title,
        metadata->>'severity'           AS severity,
        metadata->>'status'             AS status,
        metadata->>'discovered_at'      AS discovered_at,
        resolution->>'fix_sw_ref'       AS fix_sw_ref,
        resolution->>'fix_description'  AS fix_description,
        resolution->>'fix_date'         AS fix_date
    FROM issues
    WHERE id = 'iss-LLC-thrashing-0221'
""", ENGINE)

# SW profile feature_flags 비교
# PostgreSQL은 unquoted alias를 소문자로 fold하므로 "" 로 case 보존
df_flags = query_df("""
    SELECT
        id,
        metadata->>'version'                        AS version,
        metadata->>'release_date'                   AS release_date,
        feature_flags->>'LLC_per_ip_partition'      AS "LLC_per_ip_partition",
        feature_flags->>'LLC_dynamic_allocation'    AS "LLC_dynamic_allocation",
        feature_flags->>'TNR_early_abort'           AS "TNR_early_abort",
        feature_flags->>'MFC_hwae'                  AS "MFC_hwae",
        jsonb_array_length(
            COALESCE(compatibility->'breaking_changes','[]'::jsonb)
        )                                           AS breaking_change_count
    FROM sw_profiles
    ORDER BY metadata->>'version'
""", ENGINE)

# 변경된 플래그만 추출
flag_cols = ['LLC_per_ip_partition', 'LLC_dynamic_allocation',
             'TNR_early_abort', 'MFC_hwae']
row_old_f = df_flags[df_flags['id'] == 'sw-vendor-v1.2.3'].iloc[0]
row_new_f = df_flags[df_flags['id'] == 'sw-vendor-v1.3.0'].iloc[0]
changed = [c for c in flag_cols if row_old_f[c] != row_new_f[c]]

print(f'변경된 feature_flags: {changed}')
display(HTML('<h4>이슈 상세</h4>'))
display(df_issue.T.rename(columns={0: 'value'}))
display(HTML('<h4>SW Profile Feature Flags 비교</h4>'))
display(df_flags)

In [ ]:
issue = df_issue.iloc[0]

# --- 텍스트 카드 대시보드 ---
fig3, axes3 = plt.subplots(1, 3, figsize=(14, 3.5))
for ax in axes3:
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

def card(ax, title, lines, bg, title_color='white'):
    ax.add_patch(plt.Rectangle((0,0), 1, 1, transform=ax.transAxes,
                               facecolor=bg, edgecolor='#DEE2E6',
                               linewidth=1.5, clip_on=False))
    ax.text(0.5, 0.88, title, transform=ax.transAxes,
            ha='center', va='top', fontsize=11, fontweight='bold',
            color=title_color)
    ax.axhline(0.80, color=title_color, alpha=0.3, xmin=0.05, xmax=0.95,
               transform=ax.transAxes)
    y = 0.70
    for line in lines:
        color  = line.get('color', ACCENT)
        weight = line.get('weight', 'normal')
        size   = line.get('size', 9)
        ax.text(0.08, y, line['text'], transform=ax.transAxes,
                ha='left', va='top', fontsize=size,
                color=color, fontweight=weight, wrap=True)
        y -= line.get('dy', 0.14)

# 카드 1: Issue
card(axes3[0], '🐛  Issue', [
    {'text': issue['id'], 'color': '#E74C3C', 'weight': 'bold', 'size': 9},
    {'text': f'발견: {issue["discovered_at"]}', 'color': ACCENT},
    {'text': f'심각도: {issue["severity"]}', 'color': ACCENT},
    {'text': f'상태: ✓ {issue["status"]}', 'color': C_NEW, 'weight': 'bold'},
    {'text': 'LLC shared pool에서 ISP.TNR↔MFC\n경쟁 → 55°C 이상에서 thrashing',
     'color': '#7F8C8D', 'dy': 0.18},
], bg='#FEF9E7')

# 카드 2: Root Cause (feature_flags diff)
card(axes3[1], '🔍  Root Cause — feature_flags', [
    {'text': 'LLC_per_ip_partition', 'color': ACCENT, 'weight': 'bold'},
    {'text': f'  v1.2.3 :  disabled  ✗', 'color': '#E74C3C', 'weight': 'bold'},
    {'text': f'  v1.3.0 :  enabled   ✓', 'color': C_NEW,    'weight': 'bold'},
    {'text': '─────────────────────────', 'color': '#BDC3C7', 'dy': 0.12},
    {'text': issue['fix_description'],
     'color': '#7F8C8D', 'size': 8, 'dy': 0.20},
    {'text': f'Fix date: {issue["fix_date"]}',
     'color': ACCENT, 'size': 8},
], bg='#EBF5FB')

# 카드 3: Measured Impact
card(axes3[2], '📊  Measured Impact', [
    {'text': 'Total Power   2350→2150 mW', 'color': C_NEW, 'weight': 'bold'},
    {'text': '                          −200 mW  (−8.5%)', 'color': C_NEW},
    {'text': 'Peak Power    2680→2480 mW', 'color': C_NEW, 'weight': 'bold'},
    {'text': '                          −200 mW  (−7.5%)', 'color': C_NEW},
    {'text': 'Avg DDR BW      15→13 GBps', 'color': C_NEW, 'weight': 'bold'},
    {'text': '                           −2 GBps (−13.3%)', 'color': C_NEW},
    {'text': 'Latency           17→15 ms', 'color': C_NEW, 'weight': 'bold'},
    {'text': '                            −2 ms  (−11.8%)', 'color': C_NEW},
], bg='#EAFAF1')

fig3.suptitle('Root Cause Analysis — LLC Thrashing Fix',
              fontsize=12, fontweight='bold', color=ACCENT, y=1.02)
plt.tight_layout(pad=0.8)
plt.savefig('s3_root_cause_cards.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved: s3_root_cause_cards.png')

---
## Section 4. Review Gate — Before/After 승인 흐름

In [ ]:
df_review = query_df("""
    SELECT
        r.id                                                         AS review_id,
        r.gate_result,
        r.decision,
        r.approver_claim,
        r.claim_at,
        r.waiver_ref,
        w.status                                                     AS waiver_status,
        w.expires_on,
        -- evidence에서 sw_version 조회
        (SELECT sw_version_hint FROM evidence
         WHERE id = ANY(ARRAY(SELECT jsonb_array_elements_text(r.evidence_refs)))
         LIMIT 1)                                                    AS sw_version
    FROM reviews r
    LEFT JOIN waivers w ON w.id = r.waiver_ref
    ORDER BY r.id
""", ENGINE)

display(HTML('<h4>Review Gate 결과</h4>'))
display(
    df_review.style.map(
        lambda v: f'background-color:{GATE_COLORS.get(v,"")}22;'
                  f'color:{GATE_COLORS.get(v,"")}; font-weight:bold',
        subset=['gate_result']
    )
)

In [ ]:
# Plotly — Gate 결과 비교 (offline)
gate_order  = {'PASS': 2, 'WARN': 1, 'BLOCK': 0}
df_r = df_review.copy()
df_r['gate_num']   = df_r['gate_result'].map(gate_order)
df_r['label']      = df_r['sw_version'].str.replace('sw-vendor-', '')
df_r['gate_color'] = df_r['gate_result'].map(GATE_COLORS)

fig4 = go.Figure()

for _, row in df_r.iterrows():
    waiver_note = (
        f'waiver: {row["waiver_ref"]}<br>expires: {row["expires_on"]}'
        if pd.notna(row['waiver_ref']) else 'no waiver'
    )
    fig4.add_trace(go.Bar(
        x=[row['label']],
        y=[row['gate_num'] + 1],
        name=row['gate_result'],
        marker_color=row['gate_color'],
        text=f"<b>{row['gate_result']}</b><br>{row['decision']}<br>{waiver_note}",
        textposition='inside',
        insidetextanchor='middle',
        hovertemplate=(
            f"SW: {row['sw_version']}<br>"
            f"Gate: {row['gate_result']}<br>"
            f"Decision: {row['decision']}<br>"
            f"Approver: {row['approver_claim']}"
        ),
        showlegend=False,
    ))

fig4.update_layout(
    **plotly_layout('Review Gate Result — Before vs After LLC Fix'),
    yaxis=dict(
        tickvals=[1, 2, 3],
        ticktext=['BLOCK', 'WARN', 'PASS'],
        title='Gate Level',
        range=[0, 3.5],
    ),
    xaxis_title='SW Baseline',
    height=380,
    bargap=0.4,
)
fig4.show()

---
## Section 4. The Story

### Before — sw-vendor-v1.2.3 (2026-02-21 이후)

> UHD 60fps HDR10 카메라 레코딩 시나리오를 A0 silicon + sw-v1.2.3으로 시뮬레이션한 결과,
> `overall_feasibility: exploration_only` 판정이 나왔다.
> 원인은 `LLC_per_ip_partition: disabled` — LLC를 shared pool로 운영하면서
> ISP.TNR(2 MB 필요)과 MFC(1 MB 필요)가 동일 풀을 경쟁,
> 온도 55°C 이상에서 TNR이 캐시를 반복 요청하는 **thrashing**이 발생한다.

**조치**: iss-LLC-thrashing-0221 등록 → A0 + sw-v1.2.x 한정 waiver 승인 (MFA Track 3)
→ rev-sw123: `gate_result: WARN`, `decision: approved_with_waiver`

---

### After — sw-vendor-v1.3.0 (2026-04-01 출시)

> `LLC_per_ip_partition: enabled`으로 전환 — ISP.TNR 2 MB 파티션 고정 보장.
> 동일 시나리오를 sw-v1.3.0으로 재실행: violation **0건**, `overall_feasibility: production_ready`.

**결과**: rev-sw130: `gate_result: PASS`, `decision: approved`, waiver 불필요.

---

### Measured Impact

| KPI | Before (v1.2.3) | After (v1.3.0) | Delta | % |
|-----|----------------|---------------|-------|---|
| Total Power | 2,350 mW | **2,150 mW** | −200 mW | **−8.5%** |
| Peak Power | 2,680 mW | **2,480 mW** | −200 mW | **−7.5%** |
| Avg DDR BW | 15 GBps | **13 GBps** | −2 GBps | **−13.3%** |
| Peak DDR BW | 19 GBps | **17 GBps** | −2 GBps | **−10.5%** |
| Frame Latency | 17 ms | **15 ms** | −2 ms | **−11.8%** |
| Gate Result | ⚠️ WARN | ✅ **PASS** | | |
| Feasibility | exploration_only | **production_ready** | | |

> 단 하나의 feature flag(`LLC_per_ip_partition`) 전환으로
> 전력 −8.5%, 대역폭 −13.3%, 레이턴시 −11.8% 개선 —
> **이것이 SW Catalog + Evidence DB가 포착하는 가치다.**

In [ ]:
# 30초 데모용 단일 요약 대시보드
fig5 = plt.figure(figsize=(14, 6), facecolor='white')
gs = fig5.add_gridspec(2, 4, hspace=0.45, wspace=0.35)

kpis = [
    ('Total\nPower', 'mW', 2350, 2150),
    ('Peak\nPower',  'mW', 2680, 2480),
    ('Avg DDR\nBW', 'GBps', 15, 13),
    ('Frame\nLatency', 'ms', 17, 15),
]

for i, (name, unit, vb, va) in enumerate(kpis):
    ax = fig5.add_subplot(gs[0, i])
    ax.set_facecolor(BG_LIGHT)
    pct = (va - vb) / vb * 100
    ax.bar([0], [vb], color=C_OLD, width=0.4, label='v1.2.3')
    ax.bar([1], [va], color=C_NEW, width=0.4, label='v1.3.0')
    ax.set_xticks([0,1])
    ax.set_xticklabels(['v1.2.3','v1.3.0'], fontsize=8)
    ax.set_title(f'{name}\n({unit})', fontsize=9, fontweight='bold', color=ACCENT)
    ax.text(0.5, max(vb,va)*1.04, f'{pct:+.1f}%',
            ha='center', fontsize=10, fontweight='bold',
            color=C_NEW if pct < 0 else '#E74C3C',
            transform=ax.get_xaxis_transform())
    for spine in ['top','right']: ax.spines[spine].set_visible(False)

# 하단: feature_flags diff + gate result
ax_flag = fig5.add_subplot(gs[1, :2])
ax_flag.axis('off')
flag_data = [
    ['Feature Flag', 'v1.2.3', 'v1.3.0', 'Changed?'],
    ['LLC_per_ip_partition',  'disabled', 'enabled', '✓ YES'],
    ['LLC_dynamic_allocation','enabled',  'enabled', '—'],
    ['TNR_early_abort',       'enabled',  'enabled', '—'],
    ['MFC_hwae',              'enabled',  'enabled', '—'],
]
tbl = ax_flag.table(
    cellText=flag_data[1:], colLabels=flag_data[0],
    loc='center', cellLoc='center',
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 1.5)
# 변경된 행 강조
for j in range(4):
    tbl[(1, j)].set_facecolor('#FDECEA')
    tbl[(1, j)].set_text_props(color='#C0392B', fontweight='bold')
ax_flag.set_title('feature_flags Diff', fontsize=10, fontweight='bold',
                  color=ACCENT, pad=4)

ax_gate = fig5.add_subplot(gs[1, 2:])
ax_gate.axis('off')
gate_data = [
    ['', 'sw-v1.2.3', 'sw-v1.3.0'],
    ['Feasibility', 'exploration_only', 'production_ready'],
    ['Gate Result', 'WARN ⚠️',          'PASS ✅'],
    ['Decision',    'approved_with_waiver', 'approved'],
    ['Waiver',      'required',          'not needed'],
]
tbl2 = ax_gate.table(
    cellText=gate_data[1:], colLabels=gate_data[0],
    loc='center', cellLoc='center',
)
tbl2.auto_set_font_size(False)
tbl2.set_fontsize(9)
tbl2.scale(1, 1.5)
# v1.3.0 열 초록
for row_i in range(1, 5):
    tbl2[(row_i, 2)].set_facecolor('#EAFAF1')
    tbl2[(row_i, 2)].set_text_props(color='#1E8449', fontweight='bold')
ax_gate.set_title('Review Gate Result', fontsize=10, fontweight='bold',
                  color=ACCENT, pad=4)

fig5.suptitle(
    'SW Regression Story — LLC_per_ip_partition Fix\n'
    'sw-vendor-v1.2.3 (exploration_only, WARN) → v1.3.0 (production_ready, PASS)',
    fontsize=12, fontweight='bold', color=ACCENT,
)
plt.savefig('s4_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved: s4_dashboard.png  ← 30초 데모 슬라이드용')